# GSE125511 β-caryophyllene: raw-read count reanalysis

**Status:** Method notebook awaiting nine checksum-verified NCBI SRA archives, converted FASTQs, and STAR-derived integer counts. This file has no executed results yet.

The experiment contains three UFH-001 DMSO controls and three libraries at each of 20 µM and 200 µM β-caryophyllene under 1% oxygen for 16 hours. The same controls were reused from GSE123856. The two contrasts therefore share controls and cannot be counted as independent replications. The input is unstranded STAR 2.7.11b gene counts against GENCODE human release 48. Source SRA, sample, conversion, and mapping receipts are checked before modeling.

The model tests each BCP dose against the shared vehicle group within this study. Expression changes do not establish cholesterol flux, direct HMGCR binding, pathway activity, or reproducibility in HepG2. Dose and oxygen differ from the HepG2 dataset, so cross-study directional comparison remains descriptive.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pydeseq2

ROOT = Path.cwd()
PROJECT = ROOT / "research/ufh001-bcp-fpkm"
RESULTS = PROJECT / "results/raw_counts"
receipt = json.loads((RESULTS / "processing_receipt.json").read_text())
assert receipt["aligner_version"] == "2.7.11b"
for name, field in (("reprocess_star_counts.py", "pipeline_sha256"), ("sra_run_manifest.tsv", "run_manifest_sha256"), ("sample_manifest.tsv", "sample_manifest_sha256")):
    assert hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() == receipt[field], name
assert hashlib.sha256((RESULTS / "conversion_receipt.json").read_bytes()).hexdigest() == receipt["conversion_receipt_sha256"]
for name, expected in receipt["output_sha256"].items():
    assert hashlib.sha256((RESULTS / name).read_bytes()).hexdigest() == expected, name
counts = pd.read_csv(RESULTS / "gene_counts_unstranded.tsv.gz", sep="\t", index_col="gene_id")
annotation = pd.read_csv(RESULTS / "gene_annotation.tsv.gz", sep="\t", index_col="gene_id")
star_qc = pd.read_csv(RESULTS / "star_mapping_qc.tsv", sep="\t")
manifest = pd.read_csv(PROJECT / "sample_manifest.tsv", sep="\t", keep_default_na=False)
assert counts.shape[1] == 9 and len(star_qc) == len(manifest) == 9
assert set(counts.columns) == set(manifest.matrix_column) == set(star_qc.matrix_column)
assert not counts.isna().any().any() and (counts >= 0).all().all()
assert np.issubdtype(counts.to_numpy().dtype, np.integer)
assert receipt["reused_controls"] == dict(zip(manifest.loc[manifest.duplicate_of.ne(""), "geo_accession"], manifest.loc[manifest.duplicate_of.ne(""), "duplicate_of"]))
print({"genes": len(counts), "libraries": counts.shape[1], "PyDESeq2": pydeseq2.__version__, "sjdbOverhang": receipt["index_options"]["sjdbOverhang"]})


## Sample identity and quality

The pinned sample manifest assigns arm and reused-control identity. Mapping percentages, count depth, and clustering are descriptive checks. They do not create new biological replicates or establish comparability with HepG2.


In [ ]:
manifest = manifest.set_index("matrix_column").loc[counts.columns]
assert manifest.arm.value_counts().to_dict() == {"vehicle": 3, "bcp20": 3, "bcp200": 3}
assert manifest.loc[manifest.arm.eq("vehicle"), "duplicate_of"].ne("").all()
assert manifest.loc[manifest.arm.ne("vehicle"), "duplicate_of"].eq("").all()
arm = pd.Categorical(manifest.arm, categories=["vehicle", "bcp20", "bcp200"])
metadata = pd.DataFrame({"arm": arm}, index=counts.columns)
qc = star_qc.set_index("matrix_column").loc[counts.columns].copy()
qc["arm"] = manifest.arm
qc["duplicate_of"] = manifest.duplicate_of
qc["counted_reads"] = counts.sum(axis=0)
qc["detected_genes"] = (counts > 0).sum(axis=0)
for field in ("input_reads", "uniquely_mapped_reads"):
    qc[field] = pd.to_numeric(qc[field], errors="raise")
if qc["input_reads"].le(0).any() or qc["uniquely_mapped_reads"].le(0).any() or qc["counted_reads"].le(0).any():
    raise ValueError("A library has no input, uniquely mapped, or counted reads")
qc.to_csv(RESULTS / "analysis_sample_qc.tsv", sep="\t", index_label="matrix_column")
log_cpm = np.log2(counts.div(counts.sum(axis=0), axis=1) * 1_000_000 + 1)
correlations = log_cpm.corr(method="spearman")
correlations.to_csv(RESULTS / "analysis_sample_correlations.tsv", sep="\t")
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(correlations, vmin=0.8, vmax=1.0, cmap="viridis")
ax.set_xticks(range(9), correlations.columns, rotation=45, ha="right")
ax.set_yticks(range(9), correlations.index)
fig.colorbar(image, ax=ax, label="Spearman correlation of log2(CPM+1)")
fig.tight_layout()
fig.savefig(RESULTS / "analysis_sample_correlations.png", dpi=160)
plt.close(fig)
display(qc[["geo_accession", "run_accession", "arm", "duplicate_of", "input_reads", "uniquely_mapped_reads", "counted_reads", "detected_genes"]])


## Prespecified count model

Keep a gene if at least three of nine libraries have ≥10 unstranded counts. Fit one negative-binomial `~arm` model with PyDESeq2 0.5.4, then test 20 µM and 200 µM against the same vehicle group. Benjamini–Hochberg adjustment is performed within each gene-level contrast; the two contrast lists are correlated because they share controls. Small arm sizes, reused controls, and a single hypoxic condition limit interpretation.


In [ ]:
keep = counts.ge(10).sum(axis=1).ge(3)
model_counts = counts.loc[keep].T.astype(int)
if model_counts.shape[1] < 1000:
    raise ValueError("Too few genes pass the prespecified count filter")
dds = DeseqDataSet(counts=model_counts, metadata=metadata, design="~arm", n_cpus=4)
dds.deseq2()
contrast_results = {}
for dose in ("bcp20", "bcp200"):
    stats = DeseqStats(dds, contrast=["arm", dose, "vehicle"], alpha=0.05, n_cpus=4)
    stats.summary()
    result = stats.results_df.copy()
    result.index.name = "gene_id"
    result = result.join(annotation[["gene_name", "gene_type"]], how="left")
    result = result.sort_values(["padj", "pvalue"], na_position="last")
    result.to_csv(RESULTS / f"deseq2_{dose}_vs_vehicle.tsv", sep="\t")
    contrast_results[dose] = result
    print({"dose": dose, "tested_genes": len(result), "FDR_below_0.05": int(result.padj.lt(0.05).sum()), "FDR_below_0.05_and_abs_log2FC_at_least_1": int((result.padj.lt(0.05) & result.log2FoldChange.abs().ge(1)).sum())})


## Prespecified metabolic and signaling genes

The inspection list combines cholesterol-related symbols from the processed UFH-001 audit with the five HepG2 source or audit markers. Every tested row is retained regardless of significance. A missing row means the gene failed the specified filter or identifier mapping; it is not evidence of biological absence. This is a targeted comparison, not gene-set enrichment or target engagement.


In [ ]:
targets = ["HMGCR", "HMGCS1", "SQLE", "LDLR", "SREBF2", "PGF", "NR4A1", "CSF1R", "ARTN", "CACNA1E"]
target_rows = pd.concat([result[result.gene_name.isin(targets)].assign(contrast=dose) for dose, result in contrast_results.items()])
target_rows.to_csv(RESULTS / "prespecified_genes.tsv", sep="\t")
summary = {
    "study": "GSE125511",
    "contrasts": ["20 uM beta-caryophyllene vs shared DMSO controls", "200 uM beta-caryophyllene vs shared DMSO controls"],
    "model": "PyDESeq2 0.5.4 negative binomial ~arm, Wald tests, BH FDR within each contrast",
    "filter": "at least 10 raw unstranded counts in at least 3 of 9 libraries",
    "libraries_per_arm": {"vehicle": 3, "bcp20": 3, "bcp200": 3},
    "reused_controls": receipt["reused_controls"],
    "genes_in_count_matrix": int(len(counts)),
    "tested_genes": {dose: int(len(result)) for dose, result in contrast_results.items()},
    "fdr_below_0_05": {dose: int(result.padj.lt(0.05).sum()) for dose, result in contrast_results.items()},
    "fdr_below_0_05_abs_log2fc_at_least_1": {dose: int((result.padj.lt(0.05) & result.log2FoldChange.abs().ge(1)).sum()) for dose, result in contrast_results.items()},
    "prespecified_gene_symbols": targets,
    "limitations": ["Three labeled libraries per arm", "Both contrasts reuse the same DMSO controls from GSE123856", "Only one hypoxic condition and time point", "No target engagement, pathway activity, or cross-study replication established"],
}
(RESULTS / "analysis_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
display(target_rows[["contrast", "gene_name", "baseMean", "log2FoldChange", "pvalue", "padj"]])
print(summary)
